# Reading a confirm-stage verdict correctly

`tsfm-lens` treats everything measured on the public dev corpus as
*exploratory* — many comparisons are looked at, so a significant dev result
is a **hypothesis**, not a finding (`CLAUDE.md` §6.7). The `confirm` stage
tests each registered hypothesis **exactly once** against the sealed,
held-out private corpus and reports a verdict. That single re-test is the
evidence; everything upstream of it is context.

This notebook reads a real `confirmation.json` produced by
`configs/smoke.yaml` and shows what to check *before* trusting the numbers
in it — because a confirm artifact can look complete and still not be the
one-shot confirmation the doctrine describes, and the artifact itself will
tell you so if you know where to look.

In [ ]:
import json
d = json.load(open("runs/smoke/confirm/confirmation.json"))
print("n_registered:", d["n_registered"])
print("n_replicable:", d["n_replicable"])
print("n_private_series:", d["n_private_series"])
print("repeated_look:", d["repeated_look"])

n_registered: 17
n_replicable: 14
n_private_series: 120
repeated_look: True

## Stop here before reading any p-value: `repeated_look` is `True`

This is the single most important field in the whole artifact, and it is easy
to skip past on the way to the numbers that look like results.

`confirm` refuses to overwrite an existing confirmation artifact by default
— the private corpus is a **consumable** (`CLAUDE.md` §6.7): once its series
have informed a decision, re-testing against the same series is no longer a
blind held-out check. The only way to get a second read is `--force confirm`,
which the pipeline honors but loudly labels as **not** a fresh confirmation:

```
[confirm] CONFIRM: overwriting existing confirmation artifacts because
  --force confirm was passed. This is a SECOND look at the private
  benchmark -- the verdicts below are no longer a one-shot confirmation.
```

That is exactly what produced this artifact — the previous notebook in this
pair ran `--force all` (which implies `--force confirm`) to capture one
complete, fresh log for every pipeline stage in a single command. So the
verdicts below are real and were computed correctly, but they are a
**repeated look**, not the gold-standard one-shot confirmation `CLAUDE.md`
§6.7 describes as "the evidence." `CLAUDE.md`'s own `full_report_run.yaml`
config documents exactly this situation for a real multi-model run: its
header states the confirm cost explicitly because it re-tests the same
private corpus on purpose, so its confirm verdicts *demonstrate the
mechanism* rather than constitute a fresh confirmation.

**The rule this artifact is teaching by example:** always check
`repeated_look` before quoting a confirm verdict as evidence. If it is
`true`, treat the numbers as a demonstration of the confirm mechanism (or,
per `CLAUDE.md` §6.7, as a signal that the private epoch should be
regenerated before the result is used for anything real) — not as the
one-shot confirmation the doctrine is built around. A pristine first run
(no prior `confirmation.json` on disk, no `--force confirm`) would have
`repeated_look` either absent or `false`.

In [ ]:
for t in d["tests"]:
    print(f"{t['family']:16s} confirmed={t['confirmed']!s:5s} "
          f"p={t['p']:.4f}  p_holm={t['p_holm']:.4f}  "
          f"CI=[{t['lo']:.3f}, {t['hi']:.3f}]  n={t['n']}")

level_shift      confirmed=True  p=0.0067  p_holm=0.0267  CI=[0.471, 0.730]  n=20
seasonal_lf      confirmed=False p=0.3200  p_holm=0.3200  CI=[-0.053, 0.142]  n=20
spiky            confirmed=True  p=0.0133  p_holm=0.0267  CI=[0.011, 0.078]  n=20
trend            confirmed=True  p=0.0067  p_holm=0.0267  CI=[0.512, 0.818]  n=20

## Reading the per-family verdicts

Each row is one dev-registered hypothesis, re-tested on private series for
that family (`n=20` series each, `n_boot=150` per `CLAUDE.md` §6.6's
resampling discipline — the series is always the resampling unit, never a
window). `confirmed` requires **both** a Holm-corrected p below alpha (0.05)
**and** a bootstrap CI excluding zero (`CLAUDE.md` §6.5's "a strength
requires corrected p < alpha and a CI excluding zero").

`seasonal_lf` is the one family that does *not* confirm here: its CI
`[-0.053, 0.142]` straddles zero, so even though the point estimate favors
the same model the dev run favored, the private data doesn't rule out "no
real difference." **This is exactly what a confirm stage is for** — catching
a dev-stage hypothesis that doesn't hold up, rather than rubber-stamping
every exploratory finding. Note the summary line printed at the end of the
pipeline run said "3/4 dev hypotheses confirmed" — that count is these four
family-level tests, not the full 17-hypothesis registry (which also includes
L1's CKA replication and L3's per-corruption replication, both reported
separately below).

In [ ]:
c = d["cka_replication"]
print("L1 CKA replication:")
print(f"  dev CKA={c['dev_cka']:.4f}  private={c['private']['value']:.4f}  "
      f"CI=[{c['private']['lo']:.4f}, {c['private']['hi']:.4f}]  "
      f"replicates={c['replicates']}")

l3 = d["l3_replication"]
print(f"\nL3 corruption replication ({l3['n_private_series']} private series):")
for t in l3["tests"]:
    flag = "OK" if t["replicates"] else "DID NOT REPLICATE"
    print(f"  {t['corruption']:16s} dev_rho={t['dev_rho']:+.3f}  "
          f"private={t['private']['value']:+.3f}  [{flag}]")

L1 CKA replication:
  dev CKA=0.9734  private=0.9656  CI=[0.9274, 0.9798]  replicates=True

L3 corruption replication (48 private series):
  noise            dev_rho=-1.000  private=-0.778  [DID NOT REPLICATE]
  detrend          dev_rho=+0.843  private=+0.843  [OK]
  deseasonalize    dev_rho=+0.360  private=+0.301  [OK]
  frequency_shift  dev_rho=+0.087  private=-0.002  [OK]
  level_shift      dev_rho=+0.623  private=+0.430  [DID NOT REPLICATE]


## The full picture: 17 registered, 14 replicable — and what that split means

`n_replicable=14` counts hypotheses whose **evidence class supports
replication testing at all** (`ROADMAP.md` §24.7's replication roll-up,
`report/derived.py::replication_summary`) — L0's behavioral claims, L1's
geometric claim, and L3's causal-within-model fingerprints all qualify;
some claim types (L2's gain-over-baseline, clustering's AMI) don't yet have
a replication path implemented and are recorded as untestable rather than
folded into either verdict (`CLAUDE.md`'s own doctrine: never let an
untestable claim silently count as a pass or a fail).

Two corruptions — `noise` and `level_shift` — replicated in direction but
**not** within CI on this run, exactly the kind of discriminating result
that makes a replication check worth running: a rubber-stamp check would
show everything passing.

## Summary: the checklist for reading any confirm artifact

1. Check `repeated_look` first. If `true`, the artifact is a demonstration
   or a signal to regenerate the private epoch — not fresh evidence.
2. For each family test, require **both** corrected-p < alpha **and** a CI
   excluding zero before calling it confirmed — never one alone.
3. Read `l3_replication`'s per-corruption table, not just its pooled
   `overall` value — a pooled positive can average one strong agreement and
   one strong disagreement (`CLAUDE.md` §6.5's L3 stage description makes
   exactly this point about the pooled fingerprint-agreement number).
4. `n_replicable` is not "how many hypotheses were true" — it's how many had
   a replication path built for their evidence class at all.